# Stage 02 — Preprocessing & Heterogeneous Integration

**Owner:** Aadarsh Ghimire (Preprocessing & Research Design Lead)

**Objective.** Turn the 8 loaded CSVs into a **single employer-level master table** keyed on `employer_abn`. This addresses Assessment 2 rubric item 1 (preprocessing + heterogeneous integration).

**Inputs.** `data/processed/checkpoints/01_raw_data.pkl`

**Outputs.**
- `data/processed/02_master.parquet` — one row per employer with headcount + policy + external features.

**Steps.**
1. Load checkpoint from notebook 01.
2. Clean the workforce composition table (filter, normalise gender, cast headcount).
3. Aggregate headcount by gender — total workforce and management subset.
4. Aggregate promotions / resignations from `workforce_management_statistics`.
5. Pivot each questionnaire CSV into wide binary policy flags.
6. Left-join external ABS pay-gap data (if present) on `anzsic_division`.
7. Handle missing values and save the master table.

## 1. Setup and load checkpoint

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src import config, preprocessing
from src.utils import load_checkpoint

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

ckpt = load_checkpoint(config.CHECKPOINT_DIR / "01_raw_data.pkl")
data = ckpt["data"]
abs_df = ckpt["abs_df"]
print("Loaded raw dict with", len(data), "tables; external source present:", abs_df is not None)

Loaded raw dict with 8 tables; external source present: False


## 2. Clean workforce composition

Drops rows where the employer is flagged non-relevant, restricts gender to {Women, Men} (the WGEA schema uses these labels), and casts `n_employees` to int.

In [2]:
wc_raw = data["workforce_composition"]
wc = preprocessing.clean_workforce_composition(wc_raw)
print(f"workforce_composition: {len(wc_raw):>8,} → {len(wc):>8,} rows after cleaning")
wc.head()

workforce_composition:  191,129 →  191,129 rows after cleaning


,reporting_year,corporate_group_name,employer_name,employer_abn,is_relevant_employer,employer_size,anzsic_code,anzsic_division,anzsic_subdivision,anzsic_group,anzsic_class,manager_category,occupation,employment_status,employment_type,gender,n_employees
0,2024-25,1-STOP CONNECTIONS PTY LIMITED,1-STOP CONNECTIONS PTY LIMITED,5.810257e+10,True,<250,7000.0,"Professional, Scientific and Technical Services",Computer System Design and Related Services,Computer System Design and Related Services,Computer System Design and Related Services,Manager,CEOs,Full-time,Permanent,Women,1
1,2024-25,1-STOP CONNECTIONS PTY LIMITED,1-STOP CONNECTIONS PTY LIMITED,5.810257e+10,True,<250,7000.0,"Professional, Scientific and Technical Services",Computer System Design and Related Services,Computer System Design and Related Services,Computer System Design and Related Services,Manager,Other Executives and General Managers,Full-time,Permanent,Men,5
2,2024-25,1-STOP CONNECTIONS PTY LIMITED,1-STOP CONNECTIONS PTY LIMITED,5.810257e+10,True,<250,7000.0,"Professional, Scientific and Technical Services",Computer System Design and Related Services,Computer System Design and Related Services,Computer System Design and Related Services,Manager,Other managers,Full-time,Permanent,Men,15
3,2024-25,1-STOP CONNECTIONS PTY LIMITED,1-STOP CONNECTIONS PTY LIMITED,5.810257e+10,True,<250,7000.0,"Professional, Scientific and Technical Services",Computer System Design and Related Services,Computer System Design and Related Services,Computer System Design and Related Services,Manager,Other managers,Full-time,Permanent,Women,2
4,2024-25,1-STOP CONNECTIONS PTY LIMITED,1-STOP CONNECTIONS PTY LIMITED,5.810257e+10,True,<250,7000.0,"Professional, Scientific and Technical Services",Computer System Design and Related Services,Computer System Design and Related Services,Computer System Design and Related Services,Manager,Senior managers,Full-time,Permanent,Men,3


## 3. Build the employer master — headcount aggregation

Each employer contributes many rows (one per occupation × status × gender). We aggregate to a single row per employer with:
- `women_total`, `men_total` — summed across every occupational row.
- `women_mgmt`, `men_mgmt` — summed across rows where `manager_category == 'Manager'`.
- Promotions / resignations by gender, pivoted from `workforce_management_statistics`.

This employer-level table is the backbone of RQ1, RQ3, RQ4.

In [3]:
master = preprocessing.build_employer_master(data)
print("Master table shape:", master.shape)
print("Columns:", list(master.columns))
master.head()

21:40:31 | INFO    | src.preprocessing | Employer master table: 8239 employers, 12 cols
Master table shape: (8239, 12)
Columns: ['employer_abn', 'employer_name', 'employer_size', 'anzsic_division', 'women_total', 'men_total', 'women_mgmt', 'men_mgmt', 'promotion_men', 'promotion_women', 'resignation_men', 'resignation_women']


,employer_abn,employer_name,employer_size,anzsic_division,women_total,men_total,women_mgmt,men_mgmt,promotion_men,promotion_women,resignation_men,resignation_women
0,5.810257e+10,1-STOP CONNECTIONS PTY LIMITED,<250,"Professional, Scientific and Technical Services",32,71,7.0,23.0,7.0,6.0,9.0,2.0
1,2.015489e+10,101Warehousing Pty Ltd,0,"Transport, Postal and Warehousing",79,52,0.0,10.0,6.0,6.0,7.0,9.0
2,4.300006e+10,Stuart Alexander & Co Pty Ltd,<250,Wholesale Trade,60,60,17.0,25.0,11.0,7.0,6.0,5.0
3,2.710486e+10,1Step Communications Pty Ltd,<250,"Professional, Scientific and Technical Services",30,97,7.0,29.0,2.0,6.0,11.0,5.0
4,3.965543e+10,233 VICTORIA SQUARE HOTEL PTY LTD,250-499,Accommodation and Food Services,131,128,8.0,20.0,13.0,8.0,42.0,39.0


In [4]:
# Sanity: total women + men should be > 0 for every employer
headcount = master["women_total"] + master["men_total"]
print("Employers with zero headcount:", (headcount == 0).sum())
print("Median headcount:", int(headcount.median()))
print("Employer-size distribution:")
print(master["employer_size"].value_counts())

Employers with zero headcount: 0
Median headcount: 238
Employer-size distribution:
employer_size
<250         4286
250-499      1963
500-999      1013
1000-4999     841
5000+         130
0               6
Name: count, dtype: int64


## 4. Pivot questionnaires into binary policy flags

Each policy flag corresponds to a specific `question_index` (declared in `src/config.py::QUESTIONNAIRE_FEATURE_MAP`):

| question_index | feature name                          | meaning |
|----------------|---------------------------------------|---------|
| `DnI.FPS.N`    | `has_formal_dni_policy`               | Formal D&I policy/strategy |
| `EAct.Act.N`   | `took_action_on_pay_gap`              | Took action after a remuneration-gap analysis |
| `PPL.RegCarer.N` | `has_employer_paid_parental_leave`  | Employer-funded PPL beyond government scheme |
| `DV.DV.N`      | `has_domestic_violence_policy`        | Formal DV support policy |

Plus one aggregated flag `offers_flexible_work` (1 if the employer appears in the `questionnaire_flexible_work` table at all — i.e. offers ≥ 1 flexible-work option).

In [5]:
master = preprocessing.merge_questionnaires(master, data)
policy_cols = [c for c in master.columns if c.startswith(("has_", "offers_", "took_"))]
print("Policy flags attached:", policy_cols)
master[policy_cols].describe().T

21:40:32 | INFO    | src.preprocessing | After questionnaire merge: 17 cols (5 policy features added)
Policy flags attached: ['has_formal_dni_policy', 'took_action_on_pay_gap', 'has_employer_paid_parental_leave', 'has_domestic_violence_policy', 'offers_flexible_work']


,count,mean,std,min,25%,50%,75%,max
has_formal_dni_policy,1811.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
took_action_on_pay_gap,1410.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
has_employer_paid_parental_leave,2728.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
has_domestic_violence_policy,937.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
offers_flexible_work,8233.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0


In [6]:
# Adoption rates by employer size — quick sanity visualisation
adoption = master.groupby("employer_size")[policy_cols].mean().round(3)
adoption

,has_formal_dni_policy,took_action_on_pay_gap,has_employer_paid_parental_leave,has_domestic_violence_policy,offers_flexible_work
employer_size,,,,,
0,<NA>,<NA>,0.0,<NA>,1.0
1000-4999,0.0,0.0,0.0,0.0,1.0
250-499,0.0,0.0,0.0,0.0,1.0
500-999,0.0,0.0,0.0,0.0,1.0
5000+,0.0,0.0,0.0,0.0,1.0
<250,0.0,0.0,0.0,0.0,1.0


## 5. Integrate the external heterogeneous source (optional)

If `data/external/abs_gender_pay_gap_by_industry.csv` was loaded in notebook 01 this adds `industry_pay_gap` (ABS estimate) to every employer via a left-join on `anzsic_division`. This is the explicit heterogeneity the Assessment 2 rubric rewards.

In [7]:
master = preprocessing.integrate_external(master, abs_df)
if abs_df is not None:
    print("External numeric cols now in master:", [c for c in master.columns if c in abs_df.columns and c != 'anzsic_division'])
else:
    print("No external data — step was a no-op.")

21:40:32 | INFO    | src.preprocessing | No external ABS data — skipping heterogeneous integration.
No external data — step was a no-op.


## 6. Missing-value handling

- Drop employers with zero total headcount (can't compute the regression target).
- Policy flag NaNs (employer didn't answer that question) are filled with 0 = "no affirmative response", which is the interpretation the rubric expects for a compliance questionnaire.

In [8]:
before = len(master)
master = preprocessing.handle_missing(master)
print(f"Rows: {before:,} → {len(master):,}")
print("Nulls remaining per column:")
master.isna().sum().loc[lambda s: s > 0]

21:40:32 | INFO    | src.preprocessing | Dropped 0 rows with zero headcount; 8239 remain
Rows: 8,239 → 8,239
Nulls remaining per column:


Series([], dtype: int64)

## 7. Save master table

In [9]:
out = config.CHECKPOINT_DIR / "02_master.parquet"
try:
    master.to_parquet(out, index=False)
    print("Saved parquet →", out)
except Exception as e:
    out = out.with_suffix(".csv")
    master.to_csv(out, index=False)
    print("Parquet failed (", e, "); saved CSV →", out)
print("Final shape:", master.shape)
master.head()

Parquet failed ( ("Expected bytes, got a 'int' object", 'Conversion failed for column employer_size with type object') ); saved CSV → D:\CDU\Semester3\PRT564_DATA_ANALYTICS_AND_VISUALISATION\project\data\processed\checkpoints\02_master.csv
Final shape: (8239, 17)


,employer_abn,employer_name,employer_size,anzsic_division,women_total,men_total,women_mgmt,men_mgmt,promotion_men,promotion_women,resignation_men,resignation_women,has_formal_dni_policy,took_action_on_pay_gap,has_employer_paid_parental_leave,has_domestic_violence_policy,offers_flexible_work
0,5.810257e+10,1-STOP CONNECTIONS PTY LIMITED,<250,"Professional, Scientific and Technical Services",32,71,7.0,23.0,7.0,6.0,9.0,2.0,0,0,0,0,1
1,2.015489e+10,101Warehousing Pty Ltd,0,"Transport, Postal and Warehousing",79,52,0.0,10.0,6.0,6.0,7.0,9.0,0,0,0,0,1
2,4.300006e+10,Stuart Alexander & Co Pty Ltd,<250,Wholesale Trade,60,60,17.0,25.0,11.0,7.0,6.0,5.0,0,0,0,0,1
3,2.710486e+10,1Step Communications Pty Ltd,<250,"Professional, Scientific and Technical Services",30,97,7.0,29.0,2.0,6.0,11.0,5.0,0,0,0,0,1
4,3.965543e+10,233 VICTORIA SQUARE HOTEL PTY LTD,250-499,Accommodation and Food Services,131,128,8.0,20.0,13.0,8.0,42.0,39.0,0,0,0,0,1


## Summary

- Started with 8 heterogeneous CSVs; ended with **one row per employer** containing headcount, management counts, movement counts, binary policy flags, and optional external pay-gap data.
- All transformations are pure, reproducible, and implemented in `src/preprocessing.py`.

**Next:** `03_feature_engineering.ipynb` — build regression/classification targets and the encoded feature matrix.